# Numerical Optimization Methods
## Golden Section | Bisection | Fibonacci | Steepest Ascent

This notebook illustrates four classic single-variable and multi-variable optimization methods with worked examples, step-by-step iteration tables, and visualizations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from IPython.display import display

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11
})

---
## 1. Golden Section Search

**Concept:** Reduces the search interval using the golden ratio φ = (√5 − 1)/2 ≈ 0.618.  
Each iteration removes ~38.2% of the remaining interval.

**Example:** Minimize  f(x) = (x − 3)² + 2  on  [0, 6]  
True minimum: x* = 3,  f(x*) = 2

In [ ]:
def golden_section_search(f, a, b, tol=1e-6, max_iter=100):
    """
    Golden Section Search for minimizing a unimodal function f on [a, b].

    Parameters
    ----------
    f       : callable  – objective function
    a, b    : float     – search interval endpoints
    tol     : float     – convergence tolerance (interval width)
    max_iter: int       – maximum iterations

    Returns
    -------
    x_opt   : float     – approximate minimizer
    f_opt   : float     – approximate minimum value
    history : list      – iteration records
    """
    phi = (np.sqrt(5) - 1) / 2          # golden ratio ≈ 0.618
    history = []

    x1 = b - phi * (b - a)
    x2 = a + phi * (b - a)
    f1, f2 = f(x1), f(x2)

    for k in range(1, max_iter + 1):
        history.append({'Iter': k, 'a': a, 'b': b,
                        'x1': x1, 'x2': x2,
                        'f(x1)': f1, 'f(x2)': f2,
                        'Width': b - a})
        if (b - a) < tol:
            break
        if f1 < f2:
            b, x2, f2 = x2, x1, f1
            x1 = b - phi * (b - a)
            f1 = f(x1)
        else:
            a, x1, f1 = x1, x2, f2
            x2 = a + phi * (b - a)
            f2 = f(x2)

    x_opt = (a + b) / 2
    return x_opt, f(x_opt), history


# ── User-defined example ──────────────────────────────────────────────────────
f_gs = lambda x: (x - 3)**2 + 2
a_gs, b_gs = 0.0, 6.0

x_opt_gs, f_opt_gs, hist_gs = golden_section_search(f_gs, a_gs, b_gs, tol=1e-6)
print(f"Golden Section Result")
print(f"  x*  = {x_opt_gs:.8f}")
print(f"  f*  = {f_opt_gs:.8f}")
print(f"  Iterations: {len(hist_gs)}")

In [ ]:
# Iteration table (first 10 rows)
df_gs = pd.DataFrame(hist_gs)
df_gs = df_gs[['Iter','a','b','x1','x2','f(x1)','f(x2)','Width']]
display(df_gs.head(10).style
        .format({c: '{:.6f}' for c in df_gs.columns if c != 'Iter'})
        .set_caption('Golden Section – First 10 Iterations')
        .highlight_min(subset=['Width'], color='#d4edda'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: function + shrinking intervals
ax = axes[0]
xs = np.linspace(a_gs, b_gs, 400)
ax.plot(xs, f_gs(xs), 'steelblue', lw=2, label='f(x) = (x−3)² + 2')
colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, min(8, len(hist_gs))))
for i, (row, c) in enumerate(zip(hist_gs[:8], colors)):
    ax.axvspan(row['a'], row['b'], alpha=0.15, color=c)
ax.axvline(x_opt_gs, color='crimson', lw=1.5, ls='--', label=f'x* ≈ {x_opt_gs:.4f}')
ax.scatter([x_opt_gs], [f_opt_gs], color='crimson', zorder=5, s=60)
ax.set(xlabel='x', ylabel='f(x)', title='Golden Section – Interval Narrowing')
ax.legend(fontsize=9)

# Right: interval width convergence
ax2 = axes[1]
widths = [r['Width'] for r in hist_gs]
ax2.semilogy(range(1, len(widths)+1), widths, 'o-', color='steelblue', ms=4)
ax2.set(xlabel='Iteration', ylabel='Interval width (log scale)',
        title='Convergence of Interval Width')

plt.tight_layout()
plt.savefig('golden_section.png', bbox_inches='tight')
plt.show()

---
## 2. Bisection Method

**Concept:** Applied to the derivative f′(x). Finds a root (where f′ = 0) by halving the bracket at each step. Requires f′(a)·f′(b) < 0 initially.

**Example:** Minimize  f(x) = x³ − 6x² + 9x + 1  on  [0, 4]  
f′(x) = 3x² − 12x + 9 = 3(x−1)(x−3)  → minimum at x* = 3

In [ ]:
def bisection_method(df, a, b, tol=1e-6, max_iter=100):
    """
    Bisection method applied to the derivative df to find a minimum of f.

    Parameters
    ----------
    df      : callable  – derivative of the objective function
    a, b    : float     – bracket where df changes sign
    tol     : float     – convergence tolerance
    max_iter: int       – maximum iterations

    Returns
    -------
    x_opt   : float     – approximate minimizer
    history : list      – iteration records
    """
    assert df(a) * df(b) < 0, "df must have opposite signs at a and b"
    history = []

    for k in range(1, max_iter + 1):
        mid = (a + b) / 2
        dfm = df(mid)
        history.append({'Iter': k, 'a': a, 'b': b,
                        'mid': mid, "f'(mid)": dfm,
                        'Width': b - a})
        if (b - a) < tol:
            break
        if df(a) * dfm < 0:
            b = mid
        else:
            a = mid

    return (a + b) / 2, history


# ── User-defined example ──────────────────────────────────────────────────────
f_bs  = lambda x: x**3 - 6*x**2 + 9*x + 1
df_bs = lambda x: 3*x**2 - 12*x + 9
a_bs, b_bs = 2.0, 4.0   # bracket around the minimum at x=3

x_opt_bs, hist_bs = bisection_method(df_bs, a_bs, b_bs, tol=1e-6)
print(f"Bisection Result")
print(f"  x*  = {x_opt_bs:.8f}")
print(f"  f*  = {f_bs(x_opt_bs):.8f}")
print(f"  Iterations: {len(hist_bs)}")

In [ ]:
df_table_bs = pd.DataFrame(hist_bs)
display(df_table_bs.head(10).style
        .format({c: '{:.6f}' for c in df_table_bs.columns if c != 'Iter'})
        .set_caption('Bisection Method – First 10 Iterations')
        .highlight_min(subset=['Width'], color='#d4edda'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: f(x) and f'(x)
ax = axes[0]
xs = np.linspace(0, 4.5, 400)
ax.plot(xs, f_bs(xs),  'steelblue', lw=2, label="f(x) = x³−6x²+9x+1")
ax.plot(xs, df_bs(xs), 'darkorange', lw=2, ls='--', label="f'(x) = 3x²−12x+9")
ax.axhline(0, color='gray', lw=0.8)
ax.axvline(x_opt_bs, color='crimson', lw=1.5, ls=':', label=f'x* ≈ {x_opt_bs:.4f}')
ax.set(xlabel='x', ylabel='Value', title='Bisection – f(x) and f′(x)')
ax.legend(fontsize=9)

# Right: convergence of |f'(mid)|
ax2 = axes[1]
dfmids = [abs(r["f'(mid)"]) for r in hist_bs]
ax2.semilogy(range(1, len(dfmids)+1), dfmids, 's-', color='darkorange', ms=4)
ax2.set(xlabel='Iteration', ylabel="|f'(mid)| (log scale)",
        title="Convergence – |f'(midpoint)|")

plt.tight_layout()
plt.savefig('bisection.png', bbox_inches='tight')
plt.show()

---
## 3. Fibonacci Search Method

**Concept:** Uses Fibonacci numbers to position interior points. With n function evaluations, the final interval length is 1/Fₙ of the original. Slightly more efficient than golden section for a fixed budget of evaluations.

**Example:** Minimize  f(x) = x⁴ − 8x² + x + 5  on  [−3, 3]  
True minimum near x* ≈ −1.98

In [ ]:
def fibonacci_search(f, a, b, n=15):
    """
    Fibonacci Search for minimizing a unimodal function f on [a, b].

    Parameters
    ----------
    f    : callable – objective function
    a, b : float    – search interval
    n    : int      – number of Fibonacci iterations (uses F_{n+2} evaluations)

    Returns
    -------
    x_opt   : float – approximate minimizer
    f_opt   : float – approximate minimum value
    history : list  – iteration records
    """
    # Generate Fibonacci numbers up to F_{n+2}
    F = [1, 1]
    while len(F) <= n + 2:
        F.append(F[-1] + F[-2])

    history = []
    L = b - a

    x1 = a + F[n]   / F[n+2] * L
    x2 = a + F[n+1] / F[n+2] * L
    f1, f2 = f(x1), f(x2)

    for k in range(1, n + 1):
        history.append({'k': k, 'F_ratio': f'F[{n-k+2}]/F[{n-k+3}]',
                        'a': a, 'b': b,
                        'x1': x1, 'x2': x2,
                        'f(x1)': f1, 'f(x2)': f2,
                        'Width': b - a})
        if f1 > f2:
            a = x1
            x1, f1 = x2, f2
            x2 = a + F[n-k+1] / F[n-k+2] * (b - a) if (n-k+2) < len(F) else (a+b)/2
            f2 = f(x2)
        else:
            b = x2
            x2, f2 = x1, f1
            x1 = a + F[n-k] / F[n-k+1] * (b - a) if (n-k+1) < len(F) else (a+b)/2
            f1 = f(x1)

    x_opt = (a + b) / 2
    return x_opt, f(x_opt), history


# ── User-defined example ──────────────────────────────────────────────────────
f_fb = lambda x: x**4 - 8*x**2 + x + 5
a_fb, b_fb = -3.0, 0.0   # interval containing the left minimum

x_opt_fb, f_opt_fb, hist_fb = fibonacci_search(f_fb, a_fb, b_fb, n=15)
print(f"Fibonacci Search Result")
print(f"  x*  = {x_opt_fb:.8f}")
print(f"  f*  = {f_opt_fb:.8f}")
print(f"  Iterations: {len(hist_fb)}")

In [ ]:
df_fb = pd.DataFrame(hist_fb)
display(df_fb[['k','a','b','x1','x2','f(x1)','f(x2)','Width']].head(10).style
        .format({'k': '{:.0f}', **{c: '{:.6f}' for c in ['a','b','x1','x2','f(x1)','f(x2)','Width']}})
        .set_caption('Fibonacci Search – First 10 Iterations')
        .highlight_min(subset=['Width'], color='#d4edda'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
xs = np.linspace(-3, 3, 400)
ax.plot(xs, f_fb(xs), 'purple', lw=2, label='f(x) = x⁴−8x²+x+5')
# shade intervals from iterations
for i, row in enumerate(hist_fb[:8]):
    alpha = 0.05 + 0.07 * i
    ax.axvspan(row['a'], row['b'], alpha=alpha, color='purple')
ax.axvline(x_opt_fb, color='crimson', lw=1.5, ls='--', label=f'x* ≈ {x_opt_fb:.4f}')
ax.scatter([x_opt_fb], [f_opt_fb], color='crimson', zorder=5, s=60)
ax.set(xlabel='x', ylabel='f(x)', title='Fibonacci – Interval Narrowing')
ax.legend(fontsize=9)

ax2 = axes[1]
widths = [r['Width'] for r in hist_fb]
ax2.semilogy(range(1, len(widths)+1), widths, 'D-', color='purple', ms=4)
ax2.set(xlabel='Iteration k', ylabel='Interval width (log scale)',
        title='Convergence – Interval Width')

plt.tight_layout()
plt.savefig('fibonacci.png', bbox_inches='tight')
plt.show()

---
## 4. Steepest Ascent (Gradient Ascent) Method

**Concept:** Iteratively moves in the direction of the gradient ∇f to maximize (or −∇f to minimize). Update rule:  **x_{k+1} = x_k + α ∇f(x_k)**

**Example:** Maximize  f(x, y) = −2x² − y² + 4x + 6y − 3  
∇f = (−4x + 4,  −2y + 6) → True maximum: x* = 1,  y* = 3,  f* = 6

In [ ]:
def steepest_ascent(f, grad_f, x0, alpha=0.2, tol=1e-8, max_iter=200):
    """
    Steepest Ascent (Gradient Ascent) for maximizing f.

    Parameters
    ----------
    f       : callable        – objective function f(x) where x is array-like
    grad_f  : callable        – gradient function returning array
    x0      : array-like      – starting point
    alpha   : float           – step size (learning rate)
    tol     : float           – stop when ||∇f|| < tol
    max_iter: int             – maximum iterations

    Returns
    -------
    x_opt   : ndarray  – approximate maximizer
    f_opt   : float    – approximate maximum value
    history : list     – iteration records
    """
    x = np.array(x0, dtype=float)
    history = []

    for k in range(max_iter):
        g = np.array(grad_f(x), dtype=float)
        gnorm = np.linalg.norm(g)
        history.append({'Iter': k, 'x': x[0], 'y': x[1],
                        'f(x,y)': f(x), '||∇f||': gnorm,
                        'grad_x': g[0], 'grad_y': g[1]})
        if gnorm < tol:
            break
        x = x + alpha * g

    return x, f(x), history


# ── User-defined example ──────────────────────────────────────────────────────
f_sa     = lambda xy: -2*xy[0]**2 - xy[1]**2 + 4*xy[0] + 6*xy[1] - 3
grad_f_sa = lambda xy: np.array([-4*xy[0] + 4, -2*xy[1] + 6])
x0_sa = [-2.0, -2.0]    # starting point far from optimum

x_opt_sa, f_opt_sa, hist_sa = steepest_ascent(f_sa, grad_f_sa, x0_sa, alpha=0.2, tol=1e-8)
print(f"Steepest Ascent Result")
print(f"  x*  = ({x_opt_sa[0]:.8f}, {x_opt_sa[1]:.8f})")
print(f"  f*  = {f_opt_sa:.8f}")
print(f"  Iterations: {len(hist_sa)}")

In [ ]:
df_sa = pd.DataFrame(hist_sa)
display(df_sa[['Iter','x','y','f(x,y)','||∇f||','grad_x','grad_y']].head(12).style
        .format({'Iter': '{:.0f}', **{c: '{:.6f}' for c in ['x','y','f(x,y)','||∇f||','grad_x','grad_y']}})
        .set_caption('Steepest Ascent – First 12 Iterations')
        .highlight_max(subset=['f(x,y)'], color='#d4edda')
        .highlight_min(subset=['||∇f||'], color='#fff3cd'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: contour map with ascent path
ax = axes[0]
gx = np.linspace(-3, 4, 300)
gy = np.linspace(-3, 6, 300)
X, Y = np.meshgrid(gx, gy)
Z = -2*X**2 - Y**2 + 4*X + 6*Y - 3
cp = ax.contourf(X, Y, Z, levels=20, cmap='RdYlGn', alpha=0.75)
ax.contour(X, Y, Z, levels=20, colors='white', linewidths=0.4, alpha=0.5)
plt.colorbar(cp, ax=ax, label='f(x, y)')

path_x = [r['x'] for r in hist_sa]
path_y = [r['y'] for r in hist_sa]
ax.plot(path_x, path_y, 'o-', color='navy', ms=3, lw=1.5, label='Ascent path')
ax.plot(path_x[0], path_y[0], 'rs', ms=9, label='Start')
ax.plot(x_opt_sa[0], x_opt_sa[1], 'w*', ms=14, label=f'Max ({x_opt_sa[0]:.2f}, {x_opt_sa[1]:.2f})')
ax.set(xlabel='x', ylabel='y', title='Steepest Ascent – Contour Map & Path')
ax.legend(fontsize=9, loc='lower right')

# Right: dual convergence plot
ax2 = axes[1]
iters = [r['Iter'] for r in hist_sa]
fvals = [r['f(x,y)'] for r in hist_sa]
gnorms = [r['||∇f||'] for r in hist_sa]
lns1 = ax2.plot(iters, fvals,  'g-o', ms=3, label='f(x,y)')
ax2.set_ylabel('f(x, y)', color='green')
ax2.tick_params(axis='y', labelcolor='green')
ax3 = ax2.twinx()
lns2 = ax3.semilogy(iters, gnorms, 'b--s', ms=3, label='‖∇f‖')
ax3.set_ylabel('‖∇f‖ (log scale)', color='navy')
ax3.tick_params(axis='y', labelcolor='navy')
ax2.set(xlabel='Iteration', title='Convergence – f value & Gradient Norm')
lines = lns1 + lns2
ax2.legend(lines, [l.get_label() for l in lines], fontsize=9)

plt.tight_layout()
plt.savefig('steepest_ascent.png', bbox_inches='tight')
plt.show()

---
## 5. How to Use These Functions on Your Own Example

### Golden Section Search
```python
# Define your own function and interval
my_f = lambda x: np.sin(x) + 0.5*x   # any unimodal function
x_opt, f_opt, history = golden_section_search(my_f, a=0, b=5, tol=1e-8)
```

### Bisection Method
```python
my_df = lambda x: np.cos(x) + 0.5    # derivative of your function
# Ensure my_df(a) and my_df(b) have opposite signs!
x_opt, history = bisection_method(my_df, a=2, b=4, tol=1e-8)
```

### Fibonacci Search
```python
my_f = lambda x: (x - 1.5)**2        # unimodal on your interval
x_opt, f_opt, history = fibonacci_search(my_f, a=-2, b=5, n=20)
# Increase n for higher precision (uses F_{n+2} function evaluations)
```

### Steepest Ascent
```python
import numpy as np
# 2D example — maximize f(x, y) = -(x-2)**2 - (y-3)**2
my_f     = lambda xy: -(xy[0]-2)**2 - (xy[1]-3)**2
my_gradf = lambda xy: np.array([-2*(xy[0]-2), -2*(xy[1]-3)])
x_opt, f_opt, history = steepest_ascent(my_f, my_gradf, x0=[0,0], alpha=0.3)
# For MINIMIZATION, negate f and grad_f, or change: x = x - alpha * g
```

---
## 6. Summary Comparison

| Method | Dimension | Requires | Rate | Evaluations to reach tol |
|---|---|---|---|---|
| Golden Section | 1-D | f unimodal | Linear (0.618/step) | ~log(tol)/log(0.618) |
| Bisection | 1-D | f′ sign change | Linear (0.5/step) | ~log₂(L/tol) |
| Fibonacci | 1-D | f unimodal | Optimal for fixed n | n+2 evaluations |
| Steepest Ascent | n-D | ∇f exists | Linear (α-dependent) | Varies with α & condition |

**Key insight:** Fibonacci search is provably optimal for a fixed number of function evaluations; Golden Section is its asymptotic limit. Bisection acts on the derivative and halves the interval each step. Steepest Ascent extends to multiple dimensions but convergence depends heavily on step size α.